In [9]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

class MedicalTSLoader:
    """Загрузчик медицинских временных рядов с валидацией"""
    
    def __init__(self, filepath: str):
        self.filepath = filepath
        self.df = None
        self.numeric_cols = []
        self.time_cols = []
        self.id_cols = []
        
    def load(self, sep='\t') -> pd.DataFrame:
        """Загрузка с автоматическим определением типов"""
        self.df = pd.read_excel(self.filepath)
        
        # Конвертация временных колонок
        time_candidates = ['Время поступления', 'chartTime']
        for col in time_candidates:
            if col in self.df.columns:
                self.df[col] = pd.to_datetime(self.df[col], errors='coerce')
        
        self._identify_columns()
        self._validate_structure()
        return self.df
    
    def _identify_columns(self):
        """Автоматическая классификация колонок"""
        self.id_cols = ['ID визита', 'Пациент', 'Н иб']
        self.time_cols = [col for col in self.df.columns 
                         if self.df[col].dtype == 'datetime64[ns]']
        self.numeric_cols = [col for col in self.df.select_dtypes(
            include=[np.number]).columns if col not in self.id_cols]
        
    def _validate_structure(self):
        """Проверка целостности данных"""
        required = ['ID визита', 'chartTime']
        missing = [col for col in required if col not in self.df.columns]
        if missing:
            raise ValueError(f"Отсутствуют обязательные колонки: {missing}")
        
        # Сортировка по пациенту и времени
        self.df = self.df.sort_values(['ID визита', 'chartTime']).reset_index(drop=True)
        
    def get_summary(self) -> dict:
        """Базовая статистика датасета"""
        return {
            'n_visits': self.df['ID визита'].nunique(),
            'n_patients': self.df['Пациент'].nunique() if 'Пациент' in self.df.columns else None,
            'n_records': len(self.df),
            'numeric_features': len(self.numeric_cols),
            'date_range': {
                'start': self.df['chartTime'].min(),
                'end': self.df['chartTime'].max()
            },
            'missing_rate': self.df[self.numeric_cols].isnull().mean().to_dict()
        }

In [10]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import missingno as msno

class MedicalEDA:
    """Комплексный EDA для клинических временных рядов"""
    
    def __init__(self, df: pd.DataFrame, numeric_cols: list, time_col: str = 'chartTime',
                 id_col: str = 'ID визита'):
        self.df = df.copy()
        self.numeric_cols = numeric_cols
        self.time_col = time_col
        self.id_col = id_col
        
    def run_full_eda(self, output_dir: str = 'eda_reports'):
        """Запуск полного EDA с сохранением отчетов"""
        import os
        os.makedirs(output_dir, exist_ok=True)
        
        print("=" * 60)
        print("🔍 EDA: Медицинские временные ряды")
        print("=" * 60)
        
        # 1. Анализ пропусков
        self.analyze_missingness(output_dir)
        
        # 2. Распределения и выбросы
        self.analyze_distributions(output_dir)
        
        # 3. Временные паттерны
        self.analyze_temporal_patterns(output_dir)
        
        # 4. Корреляции
        self.analyze_correlations(output_dir)
        
        # 5. Стационарность
        self.check_stationarity(output_dir)
        
        print(f"\n✅ EDA завершен. Отчеты сохранены в {output_dir}/")
        
    def analyze_missingness(self, output_dir: str):
        """Анализ паттернов пропусков — критично для медицинских данных"""
        print("\n📋 Анализ пропусков...")
        
        missing_stats = self.df[self.numeric_cols].isnull().mean() * 100
        
        # Heatmap пропусков по времени
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        
        msno.matrix(self.df[self.numeric_cols], ax=axes[0], sparkline=False)
        axes[0].set_title('Matrix пропусков', fontsize=12)
        
        # Пропуски по визитам
        missing_by_visit = self.df.groupby(self.id_col)[self.numeric_cols].apply(
            lambda x: x.isnull().mean()
        ).mean()
        missing_by_visit.sort_values().plot(kind='barh', ax=axes[1], color='steelblue')
        axes[1].set_title('Средний % пропусков по визитам', fontsize=12)
        axes[1].set_xlabel('% пропусков')
        
        plt.tight_layout()
        plt.savefig(f'{output_dir}/missingness_analysis.png', dpi=300, bbox_inches='tight')
        plt.close()
        
        # Рекомендации
        high_missing = missing_stats[missing_stats > 50]
        if len(high_missing) > 0:
            print(f"⚠️  Колонки с >50% пропусков: {list(high_missing.index)}")
            print("   → Рекомендация: рассмотреть удаление или специальную импутацию")
        
        return missing_stats
    
    def analyze_distributions(self, output_dir: str):
        """Анализ распределений и клинических выбросов"""
        print("\n📈 Анализ распределений...")
        
        n_cols = len(self.numeric_cols)
        n_rows = (n_cols + 2) // 3
        fig, axes = plt.subplots(n_rows, 3, figsize=(18, n_rows * 4))
        axes = axes.flatten()
        
        outliers_summary = {}
        
        for i, col in enumerate(self.numeric_cols):
            data = self.df[col].dropna()
            if len(data) == 0:
                continue
                
            # Гистограмма + boxplot
            axes[i].hist(data, bins=30, alpha=0.7, color='steelblue', edgecolor='white')
            
            # IQR выбросы
            Q1, Q3 = data.quantile([0.25, 0.75])
            IQR = Q3 - Q1
            outliers = data[(data < Q1 - 1.5 * IQR) | (data > Q3 + 1.5 * IQR)]
            outliers_summary[col] = {
                'n_outliers': len(outliers),
                'pct_outliers': len(outliers) / len(data) * 100,
                'range': (data.min(), data.max()),
                'iqr_range': (Q1 - 1.5 * IQR, Q3 + 1.5 * IQR)
            }
            
            axes[i].axvline(Q1 - 1.5 * IQR, color='red', linestyle='--', alpha=0.5)
            axes[i].axvline(Q3 + 1.5 * IQR, color='red', linestyle='--', alpha=0.5)
            axes[i].set_title(f'{col}\nOutliers: {len(outliers)} ({len(outliers)/len(data)*100:.1f}%)', 
                             fontsize=10)
        
        for j in range(i + 1, len(axes)):
            axes[j].set_visible(False)
            
        plt.tight_layout()
        plt.savefig(f'{output_dir}/distributions.png', dpi=300, bbox_inches='tight')
        plt.close()
        
        # Вывод клинически значимых выбросов
        print("\n🚨 Выбросы по параметрам:")
        for col, stats_ in outliers_summary.items():
            if stats_['pct_outliers'] > 5:
                print(f"   {col}: {stats_['pct_outliers']:.1f}% выбросов, "
                      f"диапазон {stats_['range']}, IQR {stats_['iqr_range']}")
        
        return outliers_summary
    
    def analyze_temporal_patterns(self, output_dir: str):
        """Анализ временных паттернов и частоты измерений"""
        print("\n⏱️  Анализ временных паттернов...")
        
        fig, axes = plt.subplots(2, 2, figsize=(16, 10))
        
        # 1. Распределение длительности визитов
        visit_duration = self.df.groupby(self.id_col)[self.time_col].agg(
            lambda x: (x.max() - x.min()).total_seconds() / 3600
        )
        axes[0, 0].hist(visit_duration, bins=30, color='steelblue', edgecolor='white')
        axes[0, 0].set_xlabel('Длительность визита (часы)')
        axes[0, 0].set_ylabel('Количество визитов')
        axes[0, 0].set_title(f'Распределение длительности визитов\nMedian: {visit_duration.median():.1f}ч')
        
        # 2. Количество измерений по визитам
        measurements_per_visit = self.df.groupby(self.id_col).size()
        axes[0, 1].hist(measurements_per_visit, bins=30, color='darkorange', edgecolor='white')
        axes[0, 1].set_xlabel('Количество измерений')
        axes[0, 1].set_ylabel('Количество визитов')
        axes[0, 1].set_title(f'Измерений на визит\nMedian: {measurements_per_visit.median():.0f}')
        
        # 3. Heatmap активности по времени
        self.df['hour'] = self.df[self.time_col].dt.hour
        self.df['day_of_stay'] = self.df.groupby(self.id_col).cumcount()
        
        # Усредненная активность по часам
        hourly_activity = self.df.groupby('hour')[self.id_col].count()
        axes[1, 0].bar(hourly_activity.index, hourly_activity.values, color='forestgreen')
        axes[1, 0].set_xlabel('Час суток')
        axes[1, 0].set_ylabel('Количество измерений')
        axes[1, 0].set_title('Активность измерений по часам')
        
        # 4. Пример временного ряда для одного визита
        sample_visit = self.df[self.id_col].value_counts().idxmax()
        sample_data = self.df[self.df[self.id_col] == sample_visit]
        
        for col in self.numeric_cols[:3]:  # Первые 3 колонки
            if col in sample_data.columns:
                axes[1, 1].plot(sample_data[self.time_col], sample_data[col], 
                               marker='o', label=col, alpha=0.7)
        axes[1, 1].set_xlabel('Время')
        axes[1, 1].set_ylabel('Значение')
        axes[1, 1].set_title(f'Пример временного ряда (визит {sample_visit})')
        axes[1, 1].legend()
        axes[1, 1].tick_params(axis='x', rotation=45)
        
        plt.tight_layout()
        plt.savefig(f'{output_dir}/temporal_patterns.png', dpi=300, bbox_inches='tight')
        plt.close()
        
        # Статистика интервалов
        self.df['time_diff'] = self.df.groupby(self.id_col)[self.time_col].diff()
        interval_stats = self.df['time_diff'].dt.total_seconds() / 60  # в минутах
        
        print(f"\n📊 Статистика интервалов измерений:")
        print(f"   Median: {interval_stats.median():.1f} мин")
        print(f"   Mean: {interval_stats.mean():.1f} мин")
        print(f"   IQR: {interval_stats.quantile(0.25):.1f} - {interval_stats.quantile(0.75):.1f} мин")
        
        return {
            'visit_duration_hours': visit_duration.describe().to_dict(),
            'measurements_per_visit': measurements_per_visit.describe().to_dict(),
            'interval_minutes': interval_stats.describe().to_dict()
        }
    
    def analyze_correlations(self, output_dir: str):
        """Анализ корреляций между параметрами"""
        print("\n🔗 Анализ корреляций...")
        
        corr_matrix = self.df[self.numeric_cols].corr(method='spearman')
        
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        
        sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', 
                   center=0, ax=axes[0], square=True)
        axes[0].set_title('Spearman корреляция параметров', fontsize=12)
        
        # Сильные корреляции
        strong_corr = []
        for i in range(len(corr_matrix.columns)):
            for j in range(i + 1, len(corr_matrix.columns)):
                if abs(corr_matrix.iloc[i, j]) > 0.7:
                    strong_corr.append({
                        'var1': corr_matrix.columns[i],
                        'var2': corr_matrix.columns[j],
                        'corr': corr_matrix.iloc[i, j]
                    })
        
        if strong_corr:
            axes[1].barh(range(len(strong_corr)), 
                        [abs(x['corr']) for x in strong_corr],
                        color=['red' if x['corr'] > 0 else 'blue' for x in strong_corr])
            axes[1].set_yticks(range(len(strong_corr)))
            axes[1].set_yticklabels([f"{x['var1']} vs {x['var2']}" for x in strong_corr])
            axes[1].set_xlabel('|Корреляция|')
            axes[1].set_title('Сильные корреляции (|r| > 0.7)')
        else:
            axes[1].text(0.5, 0.5, 'Нет сильных корреляций', 
                        ha='center', va='center', transform=axes[1].transAxes)
            axes[1].set_title('Сильные корреляции')
        
        plt.tight_layout()
        plt.savefig(f'{output_dir}/correlations.png', dpi=300, bbox_inches='tight')
        plt.close()
        
        if strong_corr:
            print("\n⚠️  Сильные корреляции обнаружены:")
            for sc in strong_corr:
                print(f"   {sc['var1']} ↔ {sc['var2']}: r = {sc['corr']:.3f}")
        
        return corr_matrix
    
    def check_stationarity(self, output_dir: str):
        """Проверка стационарности — важно для выбора методов анализа [[17]]"""
        from statsmodels.tsa.stattools import adfuller
        
        print("\n📉 Проверка стационарности...")
        
        stationarity_results = {}
        
        for col in self.numeric_cols:
            data = self.df[col].dropna()
            if len(data) < 10:
                continue
            
            try:
                result = adfuller(data)
                stationarity_results[col] = {
                    'adf_statistic': result[0],
                    'p_value': result[1],
                    'is_stationary': result[1] < 0.05
                }
            except:
                stationarity_results[col] = {'error': 'Недостаточно данных'}
        
        # Визуализация
        stationary_cols = [col for col, res in stationarity_results.items() 
                         if res.get('is_stationary', False)]
        non_stationary_cols = [col for col, res in stationarity_results.items() 
                              if not res.get('is_stationary', True)]
        
        print(f"\n📊 Результаты теста ADF:")
        print(f"   Стационарные: {len(stationary_cols)}")
        print(f"   Нестационарные: {len(non_stationary_cols)}")
        
        if non_stationary_cols:
            print(f"   ⚠️  Нестационарные параметры: {non_stationary_cols}")
            print("   → Рекомендация: рассмотреть дифференцирование или детрендирование")
        
        return stationarity_results

In [11]:
from scipy import interpolate
from sklearn.preprocessing import StandardScaler

class MedicalTSCleaner:
    """Очистка медицинских временных рядов с учетом клинической специфики"""
    
    def __init__(self, df: pd.DataFrame, numeric_cols: list, 
                 time_col: str = 'chartTime', id_col: str = 'ID визита'):
        self.df = df.copy()
        self.numeric_cols = numeric_cols
        self.time_col = time_col
        self.id_col = id_col
        self.clinical_ranges = {}
        
    def set_clinical_ranges(self, ranges: dict):
        """
        Установка клинически допустимых диапазонов
        Пример: {'ЧСС': (30, 220), 'tº, C': (34, 42), 'АД': (50, 250)}
        """
        self.clinical_ranges = ranges
        
    def clip_to_clinical_ranges(self) -> pd.DataFrame:
        """Обрезка значений по клиническим диапазонам"""
        print("\n✂️  Обрезка по клиническим диапазонам...")
        
        for col, (min_val, max_val) in self.clinical_ranges.items():
            if col in self.df.columns:
                n_out = ((self.df[col] < min_val) | (self.df[col] > max_val)).sum()
                if n_out > 0:
                    print(f"   {col}: обрезано {n_out} значений вне [{min_val}, {max_val}]")
                    self.df[col] = self.df[col].clip(min_val, max_val)
        
        return self.df
    
    def remove_constant_columns(self, threshold: float = 0.99) -> list:
        """Удаление колонок с постоянными значениями"""
        print("\n🗑️  Поиск константных колонок...")
        
        removed = []
        for col in self.numeric_cols:
            data = self.df[col].dropna()
            if len(data) == 0:
                removed.append(col)
                continue
            
            most_common_freq = data.value_counts().iloc[0] / len(data)
            if most_common_freq > threshold:
                removed.append(col)
                print(f"   Удалена: {col} ({most_common_freq*100:.1f}% одно значение)")
        
        self.numeric_cols = [col for col in self.numeric_cols if col not in removed]
        self.df = self.df.drop(columns=removed, errors='ignore')
        
        return removed
    
    def handle_missing_values(self, strategy: str = 'forward_fill', 
                              max_gap_hours: float = 6) -> pd.DataFrame:
        """
        Импутация пропусков с учетом временной природы данных
        
        Strategies:
        - 'forward_fill': последующее значение вперед (с ограничением)
        - 'linear': линейная интерполяция внутри визита
        - 'median': медиана по визиту
        - 'drop': удаление строк с пропусками
        """
        print(f"\n🔧 Импутация пропусков (стратегия: {strategy})...")
        
        for col in self.numeric_cols:
            missing_before = self.df[col].isnull().sum()
            
            if strategy == 'forward_fill':
                # Forward fill с ограничением на максимальный gap
                self.df[col] = self.df.groupby(self.id_col)[col].ffill(
                    limit=int(max_gap_hours * 60)  # предполагаем минутную частоту
                )
                
            elif strategy == 'linear':
                # Линейная интерполяция по времени
                self.df[col] = self.df.groupby(self.id_col).apply(
                    lambda x: x.set_index(self.time_col)[col].interpolate(
                        method='time', limit_direction='both'
                    ).values
                ).values
                
            elif strategy == 'median':
                self.df[col] = self.df.groupby(self.id_col)[col].transform(
                    lambda x: x.fillna(x.median())
                )
                
            elif strategy == 'drop':
                self.df = self.df.dropna(subset=[col])
            
            missing_after = self.df[col].isnull().sum()
            if missing_before > 0:
                print(f"   {col}: {missing_before} → {missing_after} пропусков")
        
        return self.df
    
    def resample_time_series(self, freq: str = '1H', 
                            agg_func: str = 'mean') -> pd.DataFrame:
        """
        Ресемплинг к регулярной частоте
        Важно: медицинские данные часто нерегулярны [[15]]
        """
        print(f"\n📅 Ресемплинг к частоте {freq}...")
        
        resampled_dfs = []
        
        for visit_id, group in self.df.groupby(self.id_col):
            group = group.set_index(self.time_col)
            resampled = group[self.numeric_cols].resample(freq).agg(agg_func)
            resampled[self.id_col] = visit_id
            
            # Сохраняем мета-информацию
            for col in self.df.columns:
                if col not in self.numeric_cols and col != self.time_col:
                    resampled[col] = group[col].iloc[0]
            
            resampled_dfs.append(resampled.reset_index())
        
        self.df = pd.concat(resampled_dfs, ignore_index=True)
        print(f"   Результирующая форма: {self.df.shape}")
        
        return self.df
    
    def get_cleaned_data(self) -> pd.DataFrame:
        return self.df

In [8]:
!pip install statsmodels

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 8.2 MB/s eta 0:00:00a 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.3/233.3 kB 23.6 MB/s eta 0:00:00


In [12]:
from scipy.signal import find_peaks
from statsmodels.tsa.stattools import acf, pacf
from scipy.stats import skew, kurtosis
import numpy as np

class MedicalTSStatistics:
    """Статистический анализ медицинских временных рядов"""
    
    def __init__(self, df: pd.DataFrame, numeric_cols: list,
                 time_col: str = 'chartTime', id_col: str = 'ID визита'):
        self.df = df.copy()
        self.numeric_cols = numeric_cols
        self.time_col = time_col
        self.id_col = id_col
        self.stats_results = {}
        
    def compute_all_statistics(self) -> dict:
        """Вычисление полного набора статистик"""
        print("\n🧮 Вычисление статистик временных рядов...")
        
        for col in self.numeric_cols:
            print(f"   Анализ {col}...")
            self.stats_results[col] = self._compute_column_stats(col)
        
        return self.stats_results
    
    def _compute_column_stats(self, col: str) -> dict:
        """Вычисление статистик для одной колонки"""
        stats = {}
        
        for visit_id, group in self.df.groupby(self.id_col):
            data = group[col].dropna()
            if len(data) < 3:
                continue
            
            visit_stats = {
                # Базовые статистики
                'mean': data.mean(),
                'std': data.std(),
                'min': data.min(),
                'max': data.max(),
                'median': data.median(),
                'iqr': data.quantile(0.75) - data.quantile(0.25),
                
                # Форма распределения
                'skewness': skew(data),
                'kurtosis': kurtosis(data),
                
                # Временные характеристики
                'n_measurements': len(data),
                'duration_hours': (group[self.time_col].max() - 
                                   group[self.time_col].min()).total_seconds() / 3600,
                
                # Динамика
                'trend': self._compute_trend(data),
                'variability': self._compute_variability(data),
                'peaks': self._count_peaks(data.values),
            }
            
            # Автокорреляция (если достаточно данных)
            if len(data) >= 10:
                try:
                    acf_vals = acf(data, nlags=min(5, len(data)//2), fft=True)
                    visit_stats['autocorr_lag1'] = acf_vals[1] if len(acf_vals) > 1 else np.nan
                except:
                    visit_stats['autocorr_lag1'] = np.nan
            
            stats[visit_id] = visit_stats
        
        # Агрегация по всем визитам
        stats_df = pd.DataFrame(stats).T
        summary = {
            'per_visit': stats,
            'aggregate': {
                'mean': stats_df.mean().to_dict(),
                'std': stats_df.std().to_dict(),
                'median': stats_df.median().to_dict()
            }
        }
        
        return summary
    
    def _compute_trend(self, data: pd.Series) -> float:
        """Вычисление линейного тренда"""
        if len(data) < 3:
            return 0
        x = np.arange(len(data))
        slope, _, _, _, _ = stats.linregress(x, data.values)
        return slope
    
    def _compute_variability(self, data: pd.Series) -> float:
        """Коэффициент вариации"""
        if data.mean() == 0:
            return 0
        return data.std() / abs(data.mean())
    
    def _count_peaks(self, data: np.ndarray) -> int:
        """Подсчет пиков"""
        if len(data) < 3:
            return 0
        peaks, _ = find_peaks(data, prominence=np.std(data) * 0.5)
        return len(peaks)
    
    def export_statistics(self, filepath: str):
        """Экспорт статистик в JSON/CSV"""
        import json
        
        # Подготовка для экспорта
        export_data = {}
        for col, stats in self.stats_results.items():
            export_data[col] = {
                'aggregate': stats['aggregate'],
                'n_visits': len(stats['per_visit'])
            }
        
        with open(filepath, 'w', encoding='utf-8') as f:
            json.dump(export_data, f, indent=2, ensure_ascii=False, default=str)
        
        print(f"\n💾 Статистики сохранены в {filepath}")

In [13]:
from scipy.stats import entropy
from sklearn.preprocessing import FunctionTransformer

class MedicalTSFeatureEngineer:
    """Генерация признаков из временных рядов для классификации"""
    
    def __init__(self, df: pd.DataFrame, numeric_cols: list,
                 time_col: str = 'chartTime', id_col: str = 'ID визита'):
        self.df = df.copy()
        self.numeric_cols = numeric_cols
        self.time_col = time_col
        self.id_col = id_col
        self.features = None
        
    def extract_all_features(self) -> pd.DataFrame:
        """Извлечение полного набора признаков"""
        print("\n⚙️  Генерация признаков для классификации...")
        
        feature_dfs = []
        
        for visit_id, group in self.df.groupby(self.id_col):
            visit_features = {self.id_col: visit_id}
            
            for col in self.numeric_cols:
                data = group[col].dropna()
                if len(data) == 0:
                    continue
                
                # Статистические признаки
                visit_features.update(self._statistical_features(data, col))
                
                # Временные признаки
                visit_features.update(self._temporal_features(data, group, col))
                
                # Признаки формы
                if len(data) >= 5:
                    visit_features.update(self._shape_features(data, col))
            
            feature_dfs.append(visit_features)
        
        self.features = pd.DataFrame(feature_dfs)
        print(f"   Сгенерировано признаков: {len(self.features.columns) - 1}")
        print(f"   Количество визитов: {len(self.features)}")
        
        return self.features
    
    def _statistical_features(self, data: pd.Series, col: str) -> dict:
        """Статистические признаки"""
        prefix = f"{col}_"
        return {
            f'{prefix}mean': data.mean(),
            f'{prefix}std': data.std(),
            f'{prefix}min': data.min(),
            f'{prefix}max': data.max(),
            f'{prefix}median': data.median(),
            f'{prefix}iqr': data.quantile(0.75) - data.quantile(0.25),
            f'{prefix}skew': skew(data),
            f'{prefix}kurtosis': kurtosis(data),
            f'{prefix}cv': data.std() / abs(data.mean()) if data.mean() != 0 else 0,
            f'{prefix}range': data.max() - data.min(),
        }
    
    def _temporal_features(self, data: pd.Series, group: pd.DataFrame, col: str) -> dict:
        """Временные признаки"""
        prefix = f"{col}_"
        features = {}
        
        # Тренд
        if len(data) >= 3:
            x = np.arange(len(data))
            slope, intercept, r_value, _, _ = stats.linregress(x, data.values)
            features[f'{prefix}trend_slope'] = slope
            features[f'{prefix}trend_r2'] = r_value ** 2
        
        # Изменение от начала к концу
        if len(data) >= 2:
            features[f'{prefix}delta'] = data.iloc[-1] - data.iloc[0]
            features[f'{prefix}pct_change'] = (data.iloc[-1] - data.iloc[0]) / abs(data.iloc[0]) if data.iloc[0] != 0 else 0
        
        # Скорость изменения
        if len(data) >= 2:
            diffs = data.diff().dropna()
            features[f'{prefix}mean_diff'] = diffs.mean()
            features[f'{prefix}std_diff'] = diffs.std()
        
        return features
    
    def _shape_features(self, data: pd.Series, col: str) -> dict:
        """Признаки формы временного ряда"""
        prefix = f"{col}_"
        features = {}
        
        # Пики
        peaks, properties = find_peaks(data.values, prominence=np.std(data) * 0.3)
        features[f'{prefix}n_peaks'] = len(peaks)
        
        if len(peaks) > 0:
            features[f'{prefix}peak_mean_height'] = np.mean(properties['prominences'])
            features[f'{prefix}peak_max_height'] = np.max(properties['prominences'])
        
        # Энтропия (мера сложности)
        hist, _ = np.histogram(data, bins=10, density=True)
        hist = hist[hist > 0]
        features[f'{prefix}entropy'] = entropy(hist)
        
        return features
    
    def add_clinical_features(self, thresholds: dict) -> pd.DataFrame:
        """
        Добавление клинически значимых признаков
        thresholds: {'ЧСС': {'low': 60, 'high': 100}, ...}
        """
        print("\n🏥 Добавление клинических признаков...")
        
        for col, thresh in thresholds.items():
            if col not in self.numeric_cols:
                continue
            
            prefix = f"{col}_"
            
            for visit_id in self.features[self.id_col]:
                data = self.df[self.df[self.id_col] == visit_id][col].dropna()
                if len(data) == 0:
                    continue
                
                idx = self.features[self.features[self.id_col] == visit_id].index[0]
                
                # Время вне диапазона
                if 'low' in thresh and 'high' in thresh:
                    out_of_range = ((data < thresh['low']) | (data > thresh['high'])).sum()
                    self.features.loc[idx, f'{prefix}time_out_of_range'] = out_of_range / len(data)
                
                # Максимальное отклонение
                if 'high' in thresh:
                    self.features.loc[idx, f'{prefix}max_above_high'] = max(0, data.max() - thresh['high'])
                if 'low' in thresh:
                    self.features.loc[idx, f'{prefix}max_below_low'] = max(0, thresh['low'] - data.min())
        
        return self.features
    
    def get_feature_matrix(self) -> pd.DataFrame:
        """Получение матрицы признаков, готовой для ML"""
        if self.features is None:
            raise ValueError("Сначала вызовите extract_all_features()")
        return self.features

In [14]:
class MedicalTSPipeline:
    """Полный пайплайн обработки медицинских временных рядов"""
    
    def __init__(self, filepath: str, config: dict = None):
        self.filepath = filepath
        self.config = config or {}
        
        # Компоненты
        self.loader = None
        self.eda = None
        self.cleaner = None
        self.stats = None
        self.feature_engineer = None
        
        # Результаты
        self.df_raw = None
        self.df_cleaned = None
        self.features = None
        self.reports = {}
        
    def run(self, output_dir: str = 'pipeline_output'):
        """Запуск полного пайплайна"""
        import os
        os.makedirs(output_dir, exist_ok=True)
        
        print("=" * 70)
        print("🚀 ЗАПУСК ПАЙПЛАЙНА: Медицинские временные ряды")
        print("=" * 70)
        
        # 1. Загрузка
        print("\n" + "=" * 70)
        print("ШАГ 1: Загрузка данных")
        print("=" * 70)
        self.loader = MedicalTSLoader(self.filepath)
        self.df_raw = self.loader.load()
        summary = self.loader.get_summary()
        
        print(f"\n📊 Сводка по данным:")
        for key, value in summary.items():
            print(f"   {key}: {value}")
        
        # 2. EDA
        print("\n" + "=" * 70)
        print("ШАГ 2: Exploratory Data Analysis")
        print("=" * 70)
        self.eda = MedicalEDA(
            self.df_raw, 
            self.loader.numeric_cols,
            self.loader.time_cols[0] if self.loader.time_cols else 'chartTime',
            'ID визита'
        )
        self.eda.run_full_eda(f'{output_dir}/eda')
        
        # 3. Очистка
        print("\n" + "=" * 70)
        print("ШАГ 3: Очистка данных")
        print("=" * 70)
        self.cleaner = MedicalTSCleaner(
            self.df_raw,
            self.loader.numeric_cols,
            'chartTime',
            'ID визита'
        )
        
        # Клинические диапазоны (настройте под ваши данные!)
        clinical_ranges = {
            'ЧСС': (30, 220),
            'tº, C': (34.0, 42.0),
            'АД': (50, 250),
            'нАД': (30, 150),
        }
        self.cleaner.set_clinical_ranges(clinical_ranges)
        
        self.cleaner.clip_to_clinical_ranges()
        self.cleaner.remove_constant_columns(threshold=0.99)
        self.cleaner.handle_missing_values(strategy='forward_fill', max_gap_hours=6)
        
        self.df_cleaned = self.cleaner.get_cleaned_data()
        print(f"\n✅ Очищенные данные: {self.df_cleaned.shape}")
        self.df_cleaned.to_csv(f'{output_dir}/cleaned_data.csv', index=False)
        
        # 4. Статистический анализ
        print("\n" + "=" * 70)
        print("ШАГ 4: Статистический анализ")
        print("=" * 70)
        self.stats = MedicalTSStatistics(
            self.df_cleaned,
            self.cleaner.numeric_cols,
            'chartTime',
            'ID визита'
        )
        self.stats.compute_all_statistics()
        self.stats.export_statistics(f'{output_dir}/statistics.json')
        
        # 5. Feature Engineering
        print("\n" + "=" * 70)
        print("ШАГ 5: Генерация признаков")
        print("=" * 70)
        self.feature_engineer = MedicalTSFeatureEngineer(
            self.df_cleaned,
            self.cleaner.numeric_cols,
            'chartTime',
            'ID визита'
        )
        self.features = self.feature_engineer.extract_all_features()
        
        # Клинические пороги для дополнительных признаков
        clinical_thresholds = {
            'ЧСС': {'low': 60, 'high': 100},
            'tº, C': {'low': 36.0, 'high': 37.5},
        }
        self.feature_engineer.add_clinical_features(clinical_thresholds)
        
        self.features = self.feature_engineer.get_feature_matrix()
        self.features.to_csv(f'{output_dir}/features.csv', index=False)
        print(f"\n✅ Матрица признаков: {self.features.shape}")
        
        # 6. Финальный отчет
        print("\n" + "=" * 70)
        print("ШАГ 6: Финальный отчет")
        print("=" * 70)
        self._generate_final_report(output_dir)
        
        print("\n" + "=" * 70)
        print("✅ ПАЙПЛАЙН ЗАВЕРШЕН УСПЕШНО!")
        print("=" * 70)
        print(f"\n📁 Результаты сохранены в: {output_dir}/")
        print(f"   - cleaned_data.csv: очищенные временные ряды")
        print(f"   - features.csv: матрица признаков для классификации")
        print(f"   - statistics.json: статистический анализ")
        print(f"   - eda/: отчеты EDA с визуализациями")
        
        return self.features
    
    def _generate_final_report(self, output_dir: str):
        """Генерация финального отчета"""
        report = {
            'pipeline_status': 'completed',
            'timestamp': pd.Timestamp.now().isoformat(),
            'data_summary': {
                'n_visits': self.features['ID визита'].nunique(),
                'n_features': len(self.features.columns) - 1,
                'feature_columns': list(self.features.columns)
            },
            'recommendations': [
                "Проверьте распределение целевой переменной перед обучением",
                "Рассмотрите масштабирование признаков (StandardScaler)",
                "Для несбалансированных данных используйте stratified split",
                "Важно: используйте outcome-independent split для валидации [[15]]"
            ]
        }
        
        import json
        with open(f'{output_dir}/pipeline_report.json', 'w') as f:
            json.dump(report, f, indent=2, ensure_ascii=False)

In [15]:
# Запуск пайплайна
pipeline = MedicalTSPipeline('./data_raw/data-timeseries_test.xlsx')
features = pipeline.run(output_dir='results_timeseries')

# features готов для объединения с табличными данными и классификации!
print(f"\n📦 Матрица признаков готова: {features.shape}")
print(f"Колонки: {list(features.columns)}")

🚀 ЗАПУСК ПАЙПЛАЙНА: Медицинские временные ряды

ШАГ 1: Загрузка данных

📊 Сводка по данным:
   n_visits: 324
   n_patients: 296
   n_records: 44246
   numeric_features: 48
   date_range: {'start': Timestamp('2025-01-03 16:04:00'), 'end': Timestamp('2025-11-06 09:30:00')}
   missing_rate: {'Unnamed: 0': 0.0, 'N иб': 0.0, 'День пребывания': 0.0, 'tº, C': 0.9279934909370339, 'АД': 0.8090674863264475, 'нАД': 0.6109252813813678, 'ЧСС': 0.40139221624553634, 'Пульс': 0.4021380463770736, 'ЦВД (CVP)': 0.999796591782308, 'SpO2': 0.8016091850110745, 'ИПД/PPV': 0.9836369389323328, 'VCO2': 1.0, 'СВ (CO)': 0.9996835872169235, 'СИ (CI)': 0.9996835872169235, 'ДЗЛК': 0.999615784477693, 'ДЛА': 0.9944853772092392, 'НСВ (CCO)': 1.0, 'НСИ (CCI)': 1.0, 'СВ по Фику': 1.0, 'Ударный объем': 0.9996835872169235, 'ИУО': 0.9996835872169235, 'ССС': 0.9996835872169235, 'ИССС': 0.9997513899561542, 'ЛСС': 0.9997513899561542, 'ИЛСС': 0.9998643945215386, 'УРЛЖ': 0.9996609863038467, 'ИУРЛЖ': 0.9996609863038467, 'УРПЖ': 0